In [1]:
from behave_analysis.database.Experiments.JAL003_ex import JAL3_25aug, JAL3_1sept, JAL3_4sept, JAL3_7sept, JAL3_22aug

from behave_analysis.database.Experiments.JAL004_ex import JAL4_3rdSept, JAL4_19thSept, JAL4_11thSept, JAL4_28aug

from behave_analysis.database.Experiments.JAL005_ex import JAL005_8thSept, JAL005_21stSept, JAL005_5thSept

from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip3_18mar, JAL6_flip7_1apr, JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar

from behave_analysis.database.Experiments.JAL007_ex import JAL7_sesh8_9apr, JAL7_sesh9_16apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_23apr

from behave_analysis.database.Experiments.JAL008_ex import JAL8_flip3_7may, JAL8_flip1_25apr, JAL8_flip2_29apr, JAL8_flip4_10may, JAL8_14may

experiments_objects = [JAL3_7sept, JAL3_4sept, JAL3_1sept, JAL3_25aug, JAL3_22aug,
JAL4_3rdSept, JAL4_19thSept, JAL4_28aug, JAL4_11thSept,
JAL005_8thSept, JAL005_21stSept, # JAL005_5thSept this one doesn't flip, but can be used as first barrier appearance
JAL6_28mar, JAL6_flip4_21mar, JAL6_flip5_25mar, JAL6_flip3_18mar, JAL6_flip7_1apr,
JAL7_sesh8_9apr, JAL7_flip5_22mar, JAL7_flip2_12mar, JAL7_sesh9_16apr, JAL7_23apr,
JAL8_flip1_25apr,JAL8_flip2_29apr, JAL8_flip3_7may, JAL8_flip4_10may, JAL8_14may]

In [3]:
from behave_analysis.process.session import get_experiment 
from JR_test_scripts.tracking2features import tracking_to_features, extract_pa_across_sesh

import os
import dill as pickle
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
# fixed vars
ceph_path = r"Z:\Jasmine_Laurence\Experimental_Data"
winstor_path = r"Y:\Laurence"
save_path = r"Z:\Jasmine_Laurence\LDA_overview"
conditions = ['shelter_only','barrier_pre_flip','barrier_post_flip']
all_angles = ['hdir','hsa','h_preflipbar_a','h_postflipbar_a','h_rightbar_a','h_leftbar_a']
all_features = ['shelter','preflip_barrier','postflip_barrier','left_barrier', 'right_barrier', 'arena_bottom', 'arena_top']
settings = 'all_angles_fr'
# 'all_angles_fr','all_angles_fr_excl_prox_15cm','all_angles_subsampled_fr','all_angles_subsampled_fr_excl_prox_15cm'

In [5]:
# pulling out prediction accuracy for all sessions for a certain set of settings

sessy, all_names, avg_pa, pa = extract_pa_across_sesh(all_angles,experiments_objects, conditions, all_features, settings)

Compare to spatial efficiency in escape


In [ ]:
'''compare to ESCAPE spatial efficiency'''
se = {}
for c in conditions:
    se[c] = np.empty((len(experiments_objects)))

for s,sesh in enumerate(experiments_objects):
    # find session
    session = get_experiment(sesh)
    base_path = ceph_path
    if os.path.exists(os.path.join(winstor_path,session.processed_path)):
        base_path = winstor_path
    
    # load tracking
    track_path = os.path.join(base_path,session.processed_path,'escapes',r"escapes_obj.pkl")
    with open(track_path, "rb") as dill_file:
        esc = pickle.load(dill_file)
    for c in conditions:
        se[c][s] = np.mean([x for x, y in zip(esc.spatial_efficiency,esc.escape_condition) if y[0] == c])

In [ ]:
fig, axs = plt.subplots(1,3, figsize = (15,5))
colorz = plt.cm.hsv(np.linspace(0, 1, len(se['shelter_only'])))
# shelter only
axs[0].scatter(se['shelter_only'],pa['hsa'][:,0],s=5,c=colorz)
axs[0].set_xlabel('spatial efficiency')
axs[0].set_ylabel('hsa prediction accuracy')
axs[0].set_title('shelter_only')

# barrier_pre_flip
axs[1].scatter(se['barrier_pre_flip'],pa['h_preflipbar_a'][:,1]-pa['h_postflipbar_a'][:,1],s=5,c=colorz)
axs[1].plot(axs[1].get_xlim(),[0,0],'--k')
axs[1].set_xlabel('spatial efficiency')
axs[1].set_ylabel('preflip-postflip prediction accuracy')
axs[1].set_title('barrier_pre_flip')

# barrier_post_flip
line_object = []
for s,p,cz in zip(se['barrier_post_flip'],pa['h_postflipbar_a'][:,2]-pa['h_preflipbar_a'][:,2],colorz):
    line = axs[2].scatter(s,p,s=5,c=cz)
    line_object.append(line)
axs[2].plot(axs[2].get_xlim(),[0,0],'--k')
axs[2].set_xlabel('spatial efficiency')
axs[2].set_ylabel('postflip-preflip prediction accuracy')
axs[2].set_title('barrier_post_flip')

axs[2].legend(line_object, all_names,loc='center left', bbox_to_anchor=(1.2, 0.5),
                columnspacing=1.0, labelspacing=0.2,
                handletextpad=0.5, handlelength=1.5)


plt.tight_layout()
# plt.savefig(save_path+'/'+str('escape_spatial_efficiency_pa_diff_allsesh_'+settings+'.png'))

Comapre prediction accuracy to homings
- spatial efficiency
- number of homing (correct to target)
- fraction of homings (correct to target)

In [ ]:
'''compare to ESCAPE spatial efficiency'''
se = {}
for c in conditions:
    se[c] = np.empty((len(experiments_objects)))

for s,sesh in enumerate(experiments_objects):
    # find session
    session = get_experiment(sesh)
    base_path = ceph_path
    if os.path.exists(os.path.join(winstor_path,session.processed_path)):
        base_path = winstor_path
    
    # load tracking
    track_path = os.path.join(base_path,session.processed_path,'homings',r"homings_obj.pkl")
    with open(track_path, "rb") as dill_file:
        esc = pickle.load(dill_file)
    for c in conditions:
        se[c][s] = np.mean([x for x, y in zip(esc.spatial_efficiency,esc.escape_condition) if y[0] == c])

In [ ]:
fig, axs = plt.subplots(1,3, figsize = (15,5))
colorz = plt.cm.hsv(np.linspace(0, 1, len(se['shelter_only'])))
# shelter only
axs[0].scatter(se['shelter_only'],pa['hsa'][:,0],s=5,c=colorz)
axs[0].set_xlabel('spatial efficiency')
axs[0].set_ylabel('hsa prediction accuracy')
axs[0].set_title('shelter_only')

# barrier_pre_flip
axs[1].scatter(se['barrier_pre_flip'],pa['h_preflipbar_a'][:,1]-pa['h_postflipbar_a'][:,1],s=5,c=colorz)
axs[1].plot(axs[1].get_xlim(),[0,0],'--k')
axs[1].set_xlabel('spatial efficiency')
axs[1].set_ylabel('preflip-postflip prediction accuracy')
axs[1].set_title('barrier_pre_flip')

# barrier_post_flip
line_object = []
for s,p,cz in zip(se['barrier_post_flip'],pa['h_postflipbar_a'][:,2]-pa['h_preflipbar_a'][:,2],colorz):
    line = axs[2].scatter(s,p,s=5,c=cz)
    line_object.append(line)
axs[2].plot(axs[2].get_xlim(),[0,0],'--k')
axs[2].set_xlabel('spatial efficiency')
axs[2].set_ylabel('postflip-preflip prediction accuracy')
axs[2].set_title('barrier_post_flip')

axs[2].legend(line_object, all_names,loc='center left', bbox_to_anchor=(1.2, 0.5),
                columnspacing=1.0, labelspacing=0.2,
                handletextpad=0.5, handlelength=1.5)


plt.tight_layout()
# plt.savefig(save_path+'/'+str('homings_spatial_efficiency_pa_diff_allsesh_'+settings+'.png'))